# Interactive continental founder-spread explorer

This synthesis notebook lets you change the proposed founder date, regional population sizes, migration / parental-source fractions, within-region mixing, and recent-contact intensity, then watch the founder-pair genealogy spread across five macrocontinents plus a separate Middle East source node.

**Important:** population presets are coarse historical estimates. The migration and mixing presets are teaching sensitivity parameters, not measured Holocene rates.

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Evolution-Creation'):
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git','/content/Evolution-Creation'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e','/content/Evolution-Creation[dev]'],check=True)


In [ ]:
import json
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from evolution_creation.continental_explorer import (
    DEFAULT_COORDINATES,
    DEFAULT_REGIONS,
    first_generation_reaching,
    parent_source_matrix_from_offdiag,
    simulate_continental_explorer,
)


In [ ]:
default_path = Path('/content/Evolution-Creation/data/continental_explorer_defaults.json') if 'google.colab' in sys.modules else Path('data/continental_explorer_defaults.json')
if not default_path.exists():
    default_path = Path('../data/continental_explorer_defaults.json')
defaults = json.loads(default_path.read_text())
regions = defaults['regions']
coords = np.array([defaults['coordinates'][name] for name in regions], dtype=float)


## 1. Founder timing and global controls

The default 11,000-year date is approximately 9000 BCE. At 28 years per generation that is about 393 generations.

In [ ]:
founder_age = widgets.IntSlider(value=11000,min=1000,max=12000,step=100,description='Years ago',continuous_update=False)
generation_interval = widgets.FloatSlider(value=28.0,min=20.0,max=35.0,step=0.5,description='Years/gen',continuous_update=False)
founder_region = widgets.Dropdown(options=regions,value='Middle East',description='Founder region')
joint_children = widgets.IntSlider(value=2,min=0,max=20,step=1,description='Joint children')
late_contact_age = widgets.IntSlider(value=500,min=0,max=3000,step=50,description='Late contact')
late_multiplier = widgets.FloatSlider(value=4.0,min=1.0,max=20.0,step=0.5,description='Late ×')
migration_scale = widgets.FloatSlider(value=1.0,min=0.0,max=10.0,step=0.1,description='Migration ×')
map_metric = widgets.Dropdown(options=[('Both founders','both'),('Any founder','any'),('Mean founder DNA','genetic')],value='both',description='Map color')
display(founder_age,generation_interval,founder_region,joint_children,late_contact_age,late_multiplier,migration_scale,map_metric)


## 2. Population sizes\n\nWith **Auto start population** enabled, changing the founder date log-interpolates the start populations between dated historical anchors from the same regional series. This interpolation is only a smooth teaching preset. Turn auto mode off to enter your own values. The target preset uses the same source's 2000 CE row only to create a smooth visual growth trajectory.

In [ ]:
start_boxes = {}\ntarget_boxes = {}\nauto_population = widgets.Checkbox(value=True,description='Auto start population from founder date',indent=False)\npopulation_status = widgets.HTML()\npop_grid = widgets.GridspecLayout(7,3,width='760px')\npop_grid[0,0]=widgets.HTML('<b>Region</b>')\npop_grid[0,1]=widgets.HTML('<b>Start population</b>')\npop_grid[0,2]=widgets.HTML('<b>Target population</b>')\nfor i,name in enumerate(regions, start=1):\n    pop_grid[i,0]=widgets.HTML(name)\n    start_boxes[name]=widgets.IntText(value=int(defaults['population_preset_9000_bce']['values'][name]),layout=widgets.Layout(width='180px'))\n    target_boxes[name]=widgets.IntText(value=int(defaults['population_target_2000_ce']['values'][name]),layout=widgets.Layout(width='180px'))\n    pop_grid[i,1]=start_boxes[name]\n    pop_grid[i,2]=target_boxes[name]\n\ndef population_preset_for_years_ago(years_ago):\n    target_bce=max(0.0,float(years_ago)-2026.0)\n    anchor_map=defaults['population_anchors_bce']['anchors']\n    years=np.array(sorted(int(k) for k in anchor_map),dtype=float)\n    if target_bce <= years[0]: lo=hi=years[0]\n    elif target_bce >= years[-1]: lo=hi=years[-1]\n    else:\n        hi_i=int(np.searchsorted(years,target_bce))\n        lo,hi=years[hi_i-1],years[hi_i]\n    if lo==hi: weight=0.0\n    else: weight=(target_bce-lo)/(hi-lo)\n    out={}\n    for name in regions:\n        a=float(anchor_map[str(int(lo))][name]); b=float(anchor_map[str(int(hi))][name])\n        out[name]=int(round(np.exp((1-weight)*np.log(a)+weight*np.log(b))))\n    return target_bce,out,lo,hi\n\ndef apply_population_for_date(_=None):\n    target_bce,values,lo,hi=population_preset_for_years_ago(founder_age.value)\n    if auto_population.value:\n        for name in regions: start_boxes[name].value=values[name]\n    population_status.value=f'<b>Population preset date:</b> ~{target_bce:,.0f} BCE; log-interpolated between {lo:,.0f} and {hi:,.0f} BCE anchors.'\n\ndef _date_changed(change):\n    if auto_population.value: apply_population_for_date()\ndef _auto_changed(change):\n    if auto_population.value: apply_population_for_date()\nfounder_age.observe(_date_changed,names='value')\nauto_population.observe(_auto_changed,names='value')\ndisplay(auto_population,population_status,pop_grid)\napply_population_for_date()\n

## 3. Within-region mixing

`0` means pedigree-state frequencies do not diffuse through random intermarriage after parental pooling. `1` means full random mating inside that macroregion.

In [ ]:
mix_boxes={}
mix_row=[]
for name,value in zip(regions,defaults['mixing_strength_default']['values']):
    box=widgets.FloatSlider(value=float(value),min=0,max=1,step=.05,description=name,continuous_update=False,layout=widgets.Layout(width='420px'))
    mix_boxes[name]=box
    mix_row.append(box)
display(widgets.VBox(mix_row))


## 4. Editable parental-source matrix

Each off-diagonal cell is **percent of parental draws per generation** in the destination row coming from the source column. The diagonal is filled automatically as the local-parent fraction.

These defaults are deliberately labeled as sensitivity assumptions, not empirical migration measurements.

In [ ]:
base_offdiag=np.array(defaults['parent_source_offdiag_default']['values'],dtype=float)
matrix_boxes={}
grid=widgets.GridspecLayout(len(regions)+1,len(regions)+1,width='1040px')
grid[0,0]=widgets.HTML('<b>destination ↓ / source →</b>')
for j,name in enumerate(regions, start=1): grid[0,j]=widgets.HTML(f'<b>{name}</b>')
for i,dest in enumerate(regions, start=1):
    grid[i,0]=widgets.HTML(f'<b>{dest}</b>')
    for j,source in enumerate(regions, start=1):
        if i==j:
            grid[i,j]=widgets.HTML('<i>local auto</i>')
        else:
            b=widgets.FloatText(value=100*base_offdiag[i-1,j-1],step=.001,layout=widgets.Layout(width='110px'))
            matrix_boxes[(i-1,j-1)]=b
            grid[i,j]=b
display(grid)


## 5. Run and visualize

The world animation uses circle size for modeled regional population and color for your selected founder metric. The trajectory panel shows the key genealogical/genetic distinction directly.

In [ ]:
run_button=widgets.Button(description='Run continental simulation',button_style='success',icon='play')
reset_button=widgets.Button(description='Reset defaults',icon='refresh')
output=widgets.Output()
display(widgets.HBox([run_button,reset_button]),output)

def collect_offdiag():
    off=np.zeros((len(regions),len(regions)),dtype=float)
    for (i,j),box in matrix_boxes.items(): off[i,j]=(box.value/100.0)*migration_scale.value
    return off

def build_map_figure(result, metric):
    if metric=='both': values=result.both_founders_fraction; title='Descended from both founders'
    elif metric=='any': values=result.any_founder_fraction; title='Descended from Adam or Eve'
    else: values=result.genetic_ancestry; title='Mean founder-pair autosomal ancestry'
    max_pop=float(result.populations.max())
    frame_step=max(1,len(result.generations)//45)
    ids=list(range(0,len(result.generations),frame_step))
    if ids[-1] != len(result.generations)-1: ids.append(len(result.generations)-1)
    def node_trace(k):
        pop=result.populations[k]
        sizes=10+42*np.sqrt(pop/max_pop)
        hover=[f'<b>{regions[i]}</b><br>Population: {pop[i]:,.0f}<br>Both founders: {100*result.both_founders_fraction[k,i]:.2f}%<br>Any founder: {100*result.any_founder_fraction[k,i]:.2f}%<br>Mean founder DNA: {100*result.genetic_ancestry[k,i]:.6f}%' for i in range(len(regions))]
        return go.Scattergeo(lat=coords[:,0],lon=coords[:,1],mode='markers+text',text=regions,textposition='bottom center',hovertext=hover,hoverinfo='text',marker=dict(size=sizes,color=100*values[k],cmin=0,cmax=100,colorscale='YlOrRd',showscale=True,colorbar=dict(title='%'),line=dict(width=1,color='black')),name=title)
    base_matrix=parent_source_matrix_from_offdiag(collect_offdiag())
    edge_traces=[]
    for i in range(len(regions)):
        for j in range(i+1,len(regions)):
            rate=max(base_matrix[i,j],base_matrix[j,i])
            if rate>0:
                edge_traces.append(go.Scattergeo(lat=[coords[i,0],coords[j,0]],lon=[coords[i,1],coords[j,1]],mode='lines',line=dict(width=max(0.5,500*rate),color='rgba(90,90,90,0.35)'),hoverinfo='skip',showlegend=False))
    frames=[go.Frame(data=[node_trace(k)],traces=[len(edge_traces)],name=str(k)) for k in ids]
    fig=go.Figure(data=edge_traces+[node_trace(ids[0])],frames=frames)
    steps=[]
    for k in ids:
        yrs=int(round(result.years_before_present[k]))
        label='present' if yrs==0 else f'{yrs:,} y ago'
        steps.append(dict(method='animate',args=[[str(k)],dict(mode='immediate',frame=dict(duration=180,redraw=True),transition=dict(duration=0))],label=label))
    fig.update_layout(title=title,geo=dict(projection_type='natural earth',showland=True,landcolor='rgb(240,240,235)',showocean=True,oceancolor='rgb(225,240,250)',showcountries=True),height=610,margin=dict(l=10,r=10,t=55,b=10),updatemenus=[dict(type='buttons',showactive=False,x=.02,y=.02,buttons=[dict(label='▶ Play',method='animate',args=[None,dict(frame=dict(duration=180,redraw=True),fromcurrent=True,transition=dict(duration=0))]),dict(label='❚❚ Pause',method='animate',args=[[None],dict(mode='immediate',frame=dict(duration=0,redraw=False),transition=dict(duration=0))])])],sliders=[dict(active=0,currentvalue=dict(prefix='Time: '),pad=dict(t=25),steps=steps)])
    return fig

def build_trajectory_figure(result):
    fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.08,subplot_titles=('Genealogical fraction descended from both founders','Mean founder-pair autosomal ancestry'))
    x=result.years_before_present
    for i,name in enumerate(regions):
        fig.add_trace(go.Scatter(x=x,y=100*result.both_founders_fraction[:,i],mode='lines',name=name,legendgroup=name),row=1,col=1)
        fig.add_trace(go.Scatter(x=x,y=100*result.genetic_ancestry[:,i],mode='lines',name=name,legendgroup=name,showlegend=False),row=2,col=1)
    fig.update_xaxes(autorange='reversed',title_text='Years before present',row=2,col=1)
    fig.update_yaxes(title_text='%',range=[0,100],row=1,col=1)
    fig.update_yaxes(title_text='%',row=2,col=1)
    fig.update_layout(height=760,hovermode='x unified',title='Genealogy can spread much faster than mean DNA ancestry')
    return fig

def build_migration_heatmap(matrix):
    z=100*matrix.copy(); np.fill_diagonal(z,np.nan)
    fig=go.Figure(go.Heatmap(z=z,x=regions,y=regions,colorscale='Blues',colorbar=dict(title='% parents'),hovertemplate='Destination: %{y}<br>Source: %{x}<br>%{z:.4f}% per parental draw<extra></extra>'))
    fig.update_layout(title='Editable parental-source matrix (off-diagonal)',xaxis_title='Source region',yaxis_title='Destination region',height=520)
    return fig

def run_sim(_=None):
    with output:
        clear_output(wait=True)
        try:
            start=[start_boxes[n].value for n in regions]
            target=[target_boxes[n].value for n in regions]
            off=collect_offdiag()
            matrix=parent_source_matrix_from_offdiag(off)
            mixing=[mix_boxes[n].value for n in regions]
            result=simulate_continental_explorer(initial_population=start,target_population=target,parent_source_matrix=matrix,mixing_strength=mixing,founder_age_years=founder_age.value,generation_interval_years=generation_interval.value,founder_region=regions.index(founder_region.value),founder_pair_joint_children=joint_children.value,late_contact_age_years=late_contact_age.value,late_contact_multiplier=late_multiplier.value,region_names=regions)
            print(f'Generations: {len(result.generations)-1}')
            print(f'Present global fraction descended from both founders: {100*result.global_both_founders_fraction[-1]:.3f}%')
            print(f'Present global mean founder-pair DNA ancestry: {100*result.global_genetic_ancestry[-1]:.8f}%')
            print('\nPresent regional summary')
            for i,name in enumerate(regions):
                print(f'{name:12s}  population={result.populations[-1,i]:,.0f}  both={100*result.both_founders_fraction[-1,i]:7.3f}%  any={100*result.any_founder_fraction[-1,i]:7.3f}%  DNA={100*result.genetic_ancestry[-1,i]:.8f}%')
            for t in [.5,.9,.99]:
                g=first_generation_reaching(result,t,metric='both')
                if g is None: print(f'Global both-founder genealogy never reaches {100*t:.0f}%')
                else: print(f'Global both-founder genealogy reaches {100*t:.0f}% around {result.years_before_present[g]:,.0f} years ago (generation {g})')
            build_map_figure(result,map_metric.value).show()
            build_trajectory_figure(result).show()
            build_migration_heatmap(matrix).show()
        except Exception as exc:
            print('Simulation error:',exc)

def reset_defaults(_=None):
    founder_age.value=11000; generation_interval.value=28; founder_region.value='Middle East'; joint_children.value=2; late_contact_age.value=500; late_multiplier.value=4; migration_scale.value=1; map_metric.value='both'
    for name in regions:
        start_boxes[name].value=int(defaults['population_preset_9000_bce']['values'][name])
        target_boxes[name].value=int(defaults['population_target_2000_ce']['values'][name])
    for name,value in zip(regions,defaults['mixing_strength_default']['values']): mix_boxes[name].value=float(value)
    for (i,j),box in matrix_boxes.items(): box.value=100*base_offdiag[i,j]
    run_sim()

run_button.on_click(run_sim)
reset_button.on_click(reset_defaults)
run_sim()


## How to interpret the animation

- **Circle size** represents modeled regional population.
- **Color** is the founder metric selected above.
- Migration arrows are schematic links based on the editable parent-source matrix.
- The genealogy panel can approach 100% while mean founder DNA remains very small. That is not a contradiction; the two quantities answer different questions.

A scenario that reaches universal ancestry demonstrates compatibility under the parameters you chose. It does **not** establish that those migration or mixing parameters occurred historically.